# 카페 데이터 전처리 노트북


1부: JSON → 전처리 CSV
2부: 전처리 CSV → Kiwi 명사·불용어 → PKL

**분석 목적**  
카페 크롤 결과를 표 형태로 정리하고, 중복·결측·텍스트 정제 후 명사 추출까지 진행한다.

**입력 데이터**  
`data/cafe_only/의대증원_카페_v2.json` 및 카페 전처리 CSV

**주요 산출물**  
카페 단독 전처리 CSV/PKL, 통합 단계에서 사용할 명사 컬럼

In [6]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths
PROJECT_ROOT = setup_paths()

from project_paths import DATA_CAFE_ONLY, STOPWORDS_DIR

# 1부 — JSON → CSV


### 파일 불러오기


In [7]:
# 파일 불러오기
import pandas as pd
import re
import json

# 1. JSON 파일 로드
# 각 플랫폼별로 크롤링된 데이터를 데이터프레임으로 불러옵니다.
df_cafe = pd.read_json(DATA_CAFE_ONLY / '의대증원_카페_v2.json')

print(f"네이버 카페 데이터 개수: {len(df_cafe)}")

df_cafe.head(50)

네이버 카페 데이터 개수: 8517


,title,doc,like,comment_cnt,comment_list,img,div,ch,ch2,time,query_from,query_to
0,자사고 1.59 수시카드,None,None,None,[],0,0,naver,cafe,None,20250630,20250629
1,"격동의 최상위권 입시, 흐름을 읽어야(2025수시)",None,2,0,[],4,0,naver,cafe,2025.06.30. 04:48,20250630,20250629
2,2026학년도 의과대학 수시모집 교과 일반전형 (3),​인제대는 작년 내신성적을 반영할 때 국영수+과학 상위 2과목만 반영하는 산식으로 ...,16,6,[{'content': '잘 읽어보았습니다. 한눈에 전체 내용이 잘 들어옵니다. 부...,1,0,naver,cafe,2025.06.29. 10:11,20250630,20250629
3,추가 의대증원 이야기 나오던데..,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요?어디서 본거 같...,0,1,[{'content': '올해는 일단 유예됐는데 취소가 아니라 유예된거라 어떻게 될...,0,0,naver,cafe,2025.06.30. 22:02,20250630,20250629
4,(공유)2026 한림대 의예과 모집 현황 및 전년도 입시 결과,None,None,None,[],0,0,naver,cafe,None,20250630,20250629
5,"연세대 미래캠퍼스 의예과 3년, 12년특례 면접준비 전략",KGS 메가반 카페입니다. ​연세대 미래캠 의예과는 1단계 서류 100% + 2단계...,6,0,[],3,0,naver,cafe,2025.06.30. 18:21,20250630,20250629
6,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ㅈㄴ 떨림 지금...,0,0,naver,cafe,2025.06.29. 22:22,20250630,20250629
7,이준석 욕하는 새끼들은 양심이 뒤진거지,이준석 스스로 탈당햇나(x)​국힘당과 노인은 물론 강제로 나가라고 했잖아.​강제로...,1,0,[],1,0,naver,cafe,2025.06.29. 11:54,20250630,20250629
8,"[기사 공유] 작년 대입 결과, 어떻게 해석할까","​입시 초보맘인데, 올해 황금돼지 인원 많다, 의대 증원 백지화 등 변화가 너무 커...",3,0,[],1,0,naver,cafe,2025.06.30. 21:41,20250630,20250629
9,수시 6장... 머리터지기 시작했어요,"게시판 안내를 확인해 주세요! ※ 글쓰기 전 선검색 바랍니다 ※ 미취학글, 과외구인...",1,11,"[{'content': '전체내신나오고, 아이의 생기부가 어느정도 파악되면 객관적인...",0,0,naver,cafe,2025.06.30. 23:28,20250630,20250629


### 본문, 댓글 비어있는 행 제거


In [8]:
# 본문, 댓글 비어있는 행 제거
import pandas as pd

# apply 함수를 사용하여 각 행의 리스트 길이를 체크
zero_comment_list_cnt = df_cafe[df_cafe['comment_list'].apply(len) == 0].shape[0]
print(f"댓글 리스트가 비어있는 데이터 개수: {zero_comment_list_cnt}개")

empty_docs = df_cafe[df_cafe['doc'].isnull() | (df_cafe['doc'] == '')]

# 2. 개수 확인
print(f"본문이 비어있는 데이터 개수: {len(empty_docs)}개")


# 1. 본문도 있고 댓글 리스트도 있는 데이터만 필터링
# 조건 1: doc가 null이 아님 (notnull)
# 조건 2: doc가 빈 문자열이 아님 (!= '')
# 조건 3: comment_list의 길이가 0보다 큼 (apply len > 0)
df_filtered = df_cafe[
    (df_cafe['doc'].notnull()) & 
    (df_cafe['doc'].str.strip() != '') & 
    (df_cafe['comment_list'].apply(len) > 0) 
].copy()

# 2. 인덱스 재정렬
df_filtered.reset_index(drop=True, inplace=True)

# 3. 결과 확인
print(f"최종 필터링된 데이터 개수: {len(df_filtered)}개")
print(f"제거된 데이터(본문누락 or 댓글없음): {len(df_cafe) - len(df_filtered)}개")

댓글 리스트가 비어있는 데이터 개수: 3075개
본문이 비어있는 데이터 개수: 3025개
최종 필터링된 데이터 개수: 3762개
제거된 데이터(본문누락 or 댓글없음): 4755개


### 인덱스 재정렬


In [9]:
# 인덱스 재정렬
# 제목: title
# 본문: doc
# 페이지 좋아요 수: like
# 댓글 수: comment_cnt
# 댓글 리스트: comment_list
# comment_list = [{'comment_content': 댓글 내용,
#                  'comment_like': 댓글 공감 수,
#                   'comment_date': 댓글 작성일}]
# 채널 종류(blog/cafe): ch
# 작성일: date(형식: 2025-01-01)
# 구간: section (1-4구간)

import pandas as pd
import ast

# 원본 데이터 복사 (원본 보존용)
df_reindex = df_filtered.copy()

# ---------------------------------------------------------
# 1. 작성일(date) 포맷 변경 및 컬럼 생성
# 기존 'time' 컬럼의 "2025.06.29. 10:11" -> 앞 10자리 슬라이싱 후 '.'을 '-'로 치환
# ---------------------------------------------------------
df_reindex['date'] = df_reindex['time'].astype(str).str.slice(0, 10).str.replace('.', '-')

# ---------------------------------------------------------
# 2. 구간(section) 할당 로직
# ---------------------------------------------------------
# 정확한 비교를 위해 date를 datetime 타입으로 임시 변환
dt_series = pd.to_datetime(df_reindex['date'], errors='coerce')

def get_section(date_val):
    if pd.isna(date_val): return None
    if pd.Timestamp('2024-01-01') <= date_val <= pd.Timestamp('2024-03-31'): return 1
    elif pd.Timestamp('2024-04-01') <= date_val <= pd.Timestamp('2024-06-30'): return 2
    elif pd.Timestamp('2024-07-01') <= date_val <= pd.Timestamp('2024-12-31'): return 3
    elif pd.Timestamp('2025-01-01') <= date_val <= pd.Timestamp('2025-06-30'): return 4
    else: return 0 # 예외 구간 (설정하신 기간 외의 데이터)

df_reindex['section'] = dt_series.apply(get_section)

# ---------------------------------------------------------
# 3. comment_list 딕셔너리 키 및 날짜 포맷 수정
# ---------------------------------------------------------
def process_comments(comments_data):
    # 1. 아예 비어있는 리스트인 경우 안전하게 빈 리스트 반환
    if isinstance(comments_data, list):
        if len(comments_data) == 0:
            return []
        comments = comments_data
        
    # 2. 결측치(NaN)인 경우 (float 타입이면서 NaN일 때만 걸러냄)
    elif isinstance(comments_data, float) and pd.isna(comments_data):
        return []
        
    # 3. 문자열인 경우 (CSV에서 불러와서 '[]' 처럼 텍스트로 굳어있는 경우)
    elif isinstance(comments_data, str):
        # 텍스트 자체가 비어있거나 대괄호만 있는 경우
        if comments_data.strip() in ["", "[]", "NaN", "nan"]:
            return []
        try:
            # 문자열을 실제 리스트/딕셔너리 객체로 변환
            comments = ast.literal_eval(comments_data)
        except (ValueError, SyntaxError):
            return []
            
    # 4. 그 외의 예외적인 형태가 들어오면 빈 리스트 반환
    else:
        return []
        
    # --- 여기서부터는 정상적인 리스트 형태의 데이터 정제 ---
    new_comments = []
    for c in comments:
        # 가끔 딕셔너리가 아닌 이상한 값이 섞여 있을 때를 대비한 방어 코드
        if not isinstance(c, dict):
            continue
            
        raw_date = str(c.get('comment_date', ''))
        formatted_date = raw_date[:10].replace('.', '-') if raw_date else ''
        
        new_dict = {
            'comment_content': c.get('content', ''),
            'comment_like': c.get('like', 0),
            'comment_date': formatted_date
        }
        new_comments.append(new_dict)
        
    return new_comments

df_reindex['comment_list'] = df_reindex['comment_list'].apply(process_comments)

# ---------------------------------------------------------
# 4. 컬럼 삭제 및 이름 변경 (인덱스 정리)
# ---------------------------------------------------------
# 삭제 대상: ch(기존), img, div, query_from, query_to, time
cols_to_drop = ['ch', 'img', 'div', 'query_from', 'query_to', 'time']
df_reindex = df_reindex.drop(columns=[col for col in cols_to_drop if col in df_reindex.columns], errors='ignore')

# ch2를 ch로 변경
df_reindex = df_reindex.rename(columns={'ch2': 'ch'})

# 최종적으로 원하는 컬럼 순서로 정렬 (선택 사항)
final_cols = ['title', 'doc', 'like', 'comment_cnt', 'comment_list', 'ch', 'date', 'section']
df_reindex = df_reindex[final_cols]

print("데이터 전처리 및 구조화 완료")
df_reindex.head(5)

데이터 전처리 및 구조화 완료


,title,doc,like,comment_cnt,comment_list,ch,date,section
0,2026학년도 의과대학 수시모집 교과 일반전형 (3),​인제대는 작년 내신성적을 반영할 때 국영수+과학 상위 2과목만 반영하는 산식으로 ...,16,6,[{'comment_content': '잘 읽어보았습니다. 한눈에 전체 내용이 잘 ...,cafe,2025-06-29,4
1,추가 의대증원 이야기 나오던데..,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요?어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4
3,수시 6장... 머리터지기 시작했어요,"게시판 안내를 확인해 주세요! ※ 글쓰기 전 선검색 바랍니다 ※ 미취학글, 과외구인...",1,11,"[{'comment_content': '전체내신나오고, 아이의 생기부가 어느정도 파...",cafe,2025-06-30,4
4,고3 수시 대학 라인 문제,게시판 안내를 확인해 주세요! 🚩 고1~고3 필수! 학종 합격 가이드 👉 https...,1,2,"[{'comment_content': '같은 고민 입니다..어렵네요ㅠㅠ', 'com...",cafe,2025-06-30,4


### 텍스트 기본 정제


In [10]:
# 텍스트 기본 정제
# 불필요한 노이즈 제거

# 순서의 중요성: 반드시 텍스트 정제(HTML, 공백 제거 등)를 먼저 진행한 후 유사도 80% 검사를 해야 합니다. 공백이나 태그가 섞여 있으면 똑같은 내용의 글도 컴퓨터는 다른 글로 인식하기 때문입니다.
# SequenceMatcher 최적화: 2만 개의 본문을 모든 글과 1:1로 비교하면 연산량이 너무 많아 주피터가 뻗을 수 있습니다. 그래서 **groupby('title')**을 사용하여 "우선 제목이 같은 글들끼리만" 본문 80% 유사도 검사를 하도록 범위를 좁혀 연산 속도를 대폭 높였습니다. (연산량을 줄이기 위한 범위 제한입니다.)
# 댓글 보존: ㅋㅋ, ㅎㅎ 같은 자음만으로 이루어진 무의미한 댓글은 clean_text를 거치면 빈 문자열("")이 됩니다. if cleaned_content: 조건을 통해 이러한 쓰레기 데이터를 리스트에서 자연스럽게 날려버렸습니다.

import re
from difflib import SequenceMatcher

# ---------------------------------------------------------
# 1. 텍스트 정제 함수 정의 (조건 1, 2, 4 통합)
# ---------------------------------------------------------
import re
import pandas as pd

def clean_text(text):
    if pd.isna(text): 
        return ""
    
    text = str(text)
    
    # 1. 특정 문장 제거 (네이버 카페 기본 양식)
    text = re.sub(r'게시판 안내를 확인해\s*주세요!?', '', text)
    text = re.sub(r'이해원연구소 코어패스 인터그레이트 4월 문항 공모!?', '', text)
    text = re.sub(r'실거래가 게시글에 관한 의견을 묻습니다!?', '', text)
    text = re.sub(r'수만휘는 수험생 대학생 선생님 학부모님이 함께 모여 서로 돕고 응원하는 대한민국 최대의 대입 커뮤니티입니다 선배들의 따뜻한 조언과 회원들의 존중 속에서 함께 만들어가는 공간입니다 이용 규정 보기!?','', text)
    
    # (선택) 추가로 자주 나오는 양식문구도 이곳에 넣을 수 있습니다.
    # text = re.sub(r'※\s*미취학글,\s*과외구인.*', '', text) 
    
    # 2. 영문 시작 ~ com으로 끝나는 도메인/URL 제거
    text = re.sub(r'[a-zA-Z][a-zA-Z0-9\.\-]*com\b', '', text)
    
    # 3. 일반적인 http/https URL 제거
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 4. HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    
    # 5. 자음/모음 반복 (ㅋㅋㅋ, ㅎㅎㅎ, ㅠㅠㅠ 등) 제거
    text = re.sub(r'[ㄱ-ㅎㅏ-ㅣ]+', '', text)
    
    # 6. 특수문자 및 이모지 제거 (한글, 영문, 숫자, 공백만 남김)
    # 주의: 여기서 점(.)이 날아가기 때문에 반드시 도메인 제거(2번)를 이보다 먼저 해야 합니다.
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    
    # 7. 연속된 공백, \n, \t 등을 하나의 공백으로 치환하고 양쪽 여백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# (선택) 댓글에도 .com 링크가 있을 수 있으니 댓글도 다시 정제
# df['comment_list'] = df['comment_list'].apply(clean_comments_list) 


# ---------------------------------------------------------
# 2. comment_list 내부 텍스트 정제 함수
# ---------------------------------------------------------
def clean_comments_list(comments):
    # 1. 만약 리스트가 아니라 문자열('[...]' 형태)로 굳어있다면 리스트로 변환
    if isinstance(comments, str):
        try:
            comments = ast.literal_eval(comments)
        except (ValueError, SyntaxError):
            return comments # 변환 실패 시 원본 유지
            
    # 리스트가 아니면 그대로 반환
    if not isinstance(comments, list):
        return comments
    
    cleaned_comments = []
    for c in comments:
        if not isinstance(c, dict):
            continue
            
        # 2. [핵심 수정] 키 이름 방어 로직
        # 'comment_content'를 먼저 찾고, 없으면 원본 키인 'content'를 찾음
        original_text = c.get('comment_content', c.get('content', ''))
        
        # 텍스트 정제 함수 적용
        cleaned_content = clean_text(original_text)
        
        # 정제 후 내용이 남아있는 경우만 보존
        if cleaned_content:
            new_c = c.copy()
            # 현재 딕셔너리가 갖고 있는 키 이름에 맞춰서 업데이트
            if 'comment_content' in new_c:
                new_c['comment_content'] = cleaned_content
            else:
                new_c['content'] = cleaned_content
                
            cleaned_comments.append(new_c)
            
    return cleaned_comments
# ---------------------------------------------------------
# [실행 파트] 정제 함수 적용
# ---------------------------------------------------------

# 정제 함수 적용
df_reindex['title'] = df_reindex['title'].apply(clean_text)
df_reindex['doc'] = df_reindex['doc'].apply(clean_text)
df_reindex['comment_list'] = df_reindex['comment_list'].apply(clean_comments_list)

# 2. 내용이 너무 짧아진(노이즈만 있던) 게시글 1차 필터링
df_reindex = df_reindex[df_reindex['doc'].str.len() >= 5].reset_index(drop=True)

print(f"1단계 정제 완료!")

# ---------------------------------------------------------
# 3. 고급 중복 제거: 동일 제목 + 본문 90% 이상 일치 (조건 3)
# ---------------------------------------------------------
print("2단계: 유사도 80% 이상 중복 게시글을 탐색합니다")

def is_similar(str1, str2, threshold=0.8):
    # SequenceMatcher를 사용해 두 문자열의 유사도 비율(0.0 ~ 1.0) 계산
    return SequenceMatcher(None, str1, str2).ratio() >= threshold

indices_to_drop = set()

# 정제된 데이터프레임(df_reindex)을 기준으로 제목이 같은 그룹끼리 검사합니다.
for title, group in df_reindex.groupby('title'):
    if len(group) > 1:
        # 그룹 내 인덱스 목록
        idxs = group.index.tolist()
        
        # 인덱스끼리 1:1 비교
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                idx1, idx2 = idxs[i], idxs[j]
                
                # 이미 삭제 예정인 인덱스는 건너뜀
                if idx2 in indices_to_drop:
                    continue
                
                doc1 = df_reindex.loc[idx1, 'doc']
                doc2 = df_reindex.loc[idx2, 'doc']
                
                # 본문 유사도가 80% 이상이면 나중 글(idx2)을 삭제 대상에 추가
                if is_similar(doc1, doc2, threshold=0.8):
                    indices_to_drop.add(idx2)

# 중복 인덱스 최종 삭제
before_cnt = len(df_reindex)
df_reindex = df_reindex.drop(index=list(indices_to_drop)).reset_index(drop=True)
after_cnt = len(df_reindex)

print(f"정제 완료!")
print(f"최종 남은 유효 데이터: {after_cnt}개")
df_reindex.tail(30)

1단계 정제 완료!
2단계: 유사도 80% 이상 중복 게시글을 탐색합니다
정제 완료!
최종 남은 유효 데이터: 3726개


,title,doc,like,comment_cnt,comment_list,ch,date,section
3696,의대증원 왜 못함,대학병원 예약 몇달씩이고 수술도 잡으면 몇개월인데 기피과 늘리고 하면 되는거 아니냐,0,2,"[{'comment_content': '높으신분들이 높나 반발해서', 'commen...",cafe,2024-01-08,1
3697,의사가 미래에도,이해원연구소 코어패스 인터그레이트 4월 문항 공모 2027시대인재N 기숙 대치 목동...,1,4,[{'comment_content': '증원하는 순간 씁쓸하긴 하지만 별 수 없네요...,cafe,2024-01-08,1
3698,공유 이재명 응급이송 뒷말 무성 의대증원 정책까지 흔들,이재명 응급이송 뒷말 무성 의대증원 정책까지 흔들 이재명 응급이송 뒷말 무성 의대증...,7,1,[{'comment_content': '윤정부가 지지율 나락간데에 공공의대 의사증원...,cafe,2024-01-08,1
3699,의대 증원하면 레지TO도 그만큼 느는 건가요,이해원연구소 코어패스 인터그레이트 4월 문항 공모 갑자기 궁금해졌는데,0,6,[{'comment_content': '수도권은 줄이고 지방만늘듯 지금도 수도권 계...,cafe,2024-01-08,1
3700,이재명 때문에 이득 보는 건 의사들이 아니라 윤석열임,윤돼지 용산놈들은 의대 증원 지르긴 했지만 딱 총선용이었고 원래 보수였던 의사들하고...,8,3,[{'comment_content': '맞아요특검이 날라가도 사람들은 이재명 욕만 ...,cafe,2024-01-08,1
3701,충격 특권충 정청래에 의한 부산대병원의 굴욕,정청래 목은 민감 후유증 고려해 잘하는 곳 병원 에서 해야 부산대병원 비하 논란 더...,10,11,"[{'comment_content': '그래서 내 년 의대증원 1000명 하나요',...",cafe,2024-01-08,1
3702,의사들은 이재명을 박살내면서 윤석열에 시그널 주는 거지요,총선때까지 이재명 박살내줄테니까 쓸데없이 의대 정원 증설 그거 말꺼내서 적전 분열시...,61,9,[{'comment_content': '자알했다 재명이 재명이 죄명 하나 더 늘었다...,cafe,2024-01-08,1
3703,의대 증원 규모 예상,이해원연구소 코어패스 인터그레이트 4월 문항 공모 무기명 투표 중몇명이나 증원할까 ...,2,15,"[{'comment_content': '그냥 한 명만 늘리죠', 'comment_l...",cafe,2024-01-08,1
3704,의대증원 진짜 하냐,요즘 뉴스 돌아가는거보니까 서울대 의대 증원말고는 하나마나같은데,0,3,[{'comment_content': '걍 의대증원은 필연임의협이 눕든 말든 여론하...,cafe,2024-01-07,1
3705,북학의 김동은,조선은 무엇이 잘못되었나 인재가 없었나 왕 세종 정조 한글 창제 의정부 설립 금난전...,0,1,[{'comment_content': '핵심을 잘 파악하고 정리한 것으로 보이네 너...,cafe,2024-01-06,1


In [11]:
# 변환할 숫자형 컬럼 목록
num_cols = ['like', 'comment_cnt', 'section']

for col in num_cols:
    if col in df_reindex.columns:
        # pd.to_numeric을 통해 숫자로 변환 (변환 불가한 값은 NaN 처리 -> 0으로 채움 -> 정수형 변환)
        df_reindex[col] = pd.to_numeric(df_reindex[col], errors='coerce').fillna(0).astype(int)

print("숫자형 데이터 변환이 완료되었습니다.")

숫자형 데이터 변환이 완료되었습니다.


In [12]:
# 최종 데이터 확인
print(f"최종 전처리 완료 데이터 개수: {len(df_reindex)}개")

# CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
save_path = str(DATA_CAFE_ONLY / '의대증원_cafedata_preprocess.csv')
df_reindex.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"\n데이터가 성공적으로 저장되었습니다: {save_path}")

최종 전처리 완료 데이터 개수: 3726개

데이터가 성공적으로 저장되었습니다: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/data/cafe_only/의대증원_cafedata_preprocess.csv


# 2부 — CSV → 명사 PKL


# 형태소 분석


In [13]:
import sys
from pathlib import Path

# 프로젝트 루트: notebooks/ 하위 어느 깊이에서 실행해도 동작
_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
from project_paths import DATA_CAFE_ONLY, DATA_INTEGRATED, STOPWORDS_DIR

# 파일 불러오기
import pandas as pd
import re

df = pd.read_csv(DATA_CAFE_ONLY / '의대증원_cafedata_preprocess.csv') 

print(f"네이버 카페 데이터 개수: {len(df)}")

df.head(10)

네이버 카페 데이터 개수: 3726


,title,doc,like,comment_cnt,comment_list,ch,date,section
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4
3,수시 6장 머리터지기 시작했어요,글쓰기 전 선검색 바랍니다 미취학글 과외구인글 아이디 추천글 댓글 달린 글 삭제시 ...,1,11,[{'comment_content': '전체내신나오고 아이의 생기부가 어느정도 파악...,cafe,2025-06-30,4
4,고3 수시 대학 라인 문제,고1 고3 필수 학종 합격 가이드 1 학생 학부모 여부 본인및 자녀의 학년 예 고3...,1,2,"[{'comment_content': '같은 고민 입니다 어렵네요', 'commen...",cafe,2025-06-30,4
5,약대도 증원소식이 있나요,약대도 증원된다만다 이야기가 있고 2026년 대비 전략 이런 글들이 많던데약대 내년...,0,1,[{'comment_content': '증원얘기는 따로 없는것 같던데 예전에 약대 ...,cafe,2025-06-30,4
6,수박먹고 대학간다 기본편 2026,이번 중위권 간담뢰 다녀와서 본격적으로 공부하려 책을 펼칩니다 아이 등급에 따라 여...,10,15,"[{'comment_content': '보고 계신 건 기본편인거죠 제목 미스매치',...",cafe,2025-06-30,4
7,재수생 학부모 계신가요 궁금한게 있는데,1 게시글과 덧글은 성의있고 예의바르게 작성하도록 합니다 게시글은 PC로 최소 7줄...,1,10,[{'comment_content': '에고 전직 삼수생으로써 6평 잘봤다고 풀어지...,cafe,2025-06-30,4
8,보건복지부장관 정은경 지명됐네요,정회원 등업대기 공간입니다 카페 취지에 맞지 않는 글은 임의 삭제 될 수 있습니다 ...,6,45,"[{'comment_content': '인사청문회에서 물어뜯기면 어떡하죠', 'co...",cafe,2025-06-29,4
9,목차,1 시계편 전쟁은 계산이다 2024년 2월 6일 의대증원 정책이 발표되었고 20일 ...,4,4,[{'comment_content': '오 손자병법이었나요 어려운 내용을 겪어낸 일...,cafe,2025-06-27,4


In [14]:
from kiwipiepy import Kiwi
from tqdm import tqdm


# 인스턴스 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

title_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
title_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(df))) :
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(df['title'][i])

    # 형태소 분석 ( Kiwi는 tokenize를 사룔)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장
    pos_result = [(t.form, t.tag) for t in tokens]
    title_token_list.append(pos_result)

    # 명사 추출
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.form) > 1]
    title_token_noun.append(nouns)

print(title_token_noun[:10])

Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
100%|██████████| 3726/3726 [00:02<00:00, 1586.70it/s]

[['학년도', '의과', '대학', '수시', '모집', '교과', '일반', '전형'], ['추가', '의대', '증원', '이야기'], ['질문'], ['수시', '머리', '시작'], ['수시', '대학', '라인', '문제'], ['약대', '증원', '소식'], ['수박', '대학', '기본'], ['재수', '학부모'], ['보건', '복지', '장관', '정은경', '지명'], ['목차']]


In [15]:
# 본문 명사 추출

# 인스턴스(일꾼) 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

document_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
document_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(df))) :
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(df['doc'][i])

    # 형태소 분석 ( Kiwi는 tokenize를 사룔)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장
    pos_result = [(t.form, t.tag) for t in tokens]
    document_token_list.append(pos_result)

    # 명사 추출
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.form) > 1]
    document_token_noun.append(nouns)
     
print(document_token_noun[:10])

Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
100%|██████████| 3726/3726 [00:49<00:00, 75.24it/s] 

[['인제대', '작년', '내신', '성적', '반영', '국영수', '과학', '상위', '과목', '반영', '학교', '내신', '반영', '방식', '특이', '학교', '직접', '비교', '대학', '공지', '작년', '기준', '인제대', '등급', '만점', '기준', '경우', '국영수', '과목', '반영', '대학', '일반', '등급', '정도', '해당', '국영', '수사과', '과목', '반영', '대학', '일반', '등급', '정도', '해당', '내신', '반대', '올해', '국영수', '과목', '정상', '반영', '수치', '내신', '등급', '정도', '상황', '의대', '정원', '낙관', '인제대', '면접', '통과', '이후', '면접', '당락', '반면', '사실', '내신', '중구난방', '예전', '면접', '최근', '면접', '배수', '이후', '면접', '당락', '인제대', '기본', '면접', '중요', '대학', '인하대', '경우', '작년', '정원', '와중', '작년', '등급', '탐구', '평균', '소수점', '최저', '학력', '등급', '올해', '탐구', '소수점', '정원', '정원', '최저', '상황', '작년', '수능', '애매', '교과', '성적', '학생', '안전빵', '인하대', '관동대', '건양', '경상대', '조선대', '강원대', '정도', '최저', '충족', '전제', '내신', '안전', '합격', '정도', '인하대', '확률', '등급', '사이', '방어', '인하대', '작년', '정시', '과학', '가산점', '만점', '감산점', '방식', '작정', '방어', '올인', '개인', '주관', '평가', '그동안', '인하대', '입학처', '모습', '인칭', '주인공', '시점', '입학', '전형', '설계', '간판', '학과', '의대', '다군', '펑크', '작년', '정시', '수시', '최저', '탐구',

In [16]:
import ast
from tqdm import tqdm

comment_token_list = [] # 형태소 전체 결과 (단어, 태그) 리스트
comment_token_noun = [] # 2글자 이상 명사만 담는 리스트

for i in tqdm(range(len(df))):
    raw_comments = df['comment_list'][i]
    
    # 해당 게시글의 모든 댓글 결과를 합칠 임시 리스트
    post_full_tokens = []
    post_nouns_only_noun = []
    post_nouns_only_verb = []
    post_nouns_only_adj = []
    
    # 1. 데이터 타입 방어 (문자열일 경우 리스트로 변환)
    if isinstance(raw_comments, str):
        try:
            comments = ast.literal_eval(raw_comments)
        except:
            comments = []
    else:
        comments = raw_comments
        
    # 2. 분석 시작
    if isinstance(comments, list):
        for c in comments:
            if isinstance(c, dict):
                # 키 이름 확인 (comment_content)
                text = str(c.get('comment_content', ''))
                
                if text.strip():
                    tokens = kiwi.tokenize(text)
                    
                    for t in tokens:
                        # (1) 전체 결과 저장 (단어, 태그) 튜플 형태
                        post_full_tokens.append((t.form, t.tag))
                        
                        # (2) 명사 필터링 (2글자 이상 NNG, NNP)
                        if t.tag in ['NNG', 'NNP'] and len(t.form) > 1:
                            post_nouns_only_noun.append(t.form)
    
    # 게시글 단위로 최종 리스트에 추가
    comment_token_list.append(post_full_tokens)
    comment_token_noun.append(post_nouns_only_noun)

print(comment_token_noun[:10])

100%|██████████| 3726/3726 [01:23<00:00, 44.84it/s] 

[['한눈', '전체', '내용', '부연', '설명', '감사', '자신', '객관', '인칭', '시점', '보믄', '파라미터', '객관', '도움', '감사', '파라미터', '선생', '고퀄', '칼럼', '감사', '휴일', '감사', '파라미터', '칼럼', '정독', '재미', '의대', '교과', '천국', '분석', '감사', '데이터', '기반', '분석'], ['올해', '유예', '취소', '유예', '회의', '내년', '증원', '결정'], ['기준', '수능', '생각', '스카이', '유리', '생각', '이유', '성적', '유리', '결국', '수미', '잡이', '수능', '답변', '감사'], ['전체', '내신', '아이', '생기', '정도', '파악', '객관', '아이', '현재', '위치', '아이', '성향', '종합', '교과', '논술', '정시', '학교', '학과', '작년', '입시', '결과', '참고', '작년', '의대', '증원', '평년', '수시', '정시', '영향', '아이', '적합', '케이스', '만족', '결과', '응원', '감사', '고민', '시기', '심사숙고', '결과', '기대', '고민', '감사', '고생', '대입', '원서', '지금', '감사', '수시', '설명회', '참고', '정도', '기말', '방학', '엄마', '시작', '여기저기', '수시', '박람회', '예약', '컨설팅', '설명회', '컨설팅', '머리', '학교', '느낌'], ['고민', '안녕', '회원님유니브클래스', '카페', '회원', '축하', '처음', '유니브클래스', '카페', '아래', '추천', '활동', '기대', '수험', '작년', '대비', '증가', '의대', '정원', '이전', '규모', '격변', '학년도', '입시', '대비', '작년', '대비', '분량', '자료집', '등급', '계열', '학교', '지원', '전략', '분석', '수시', '전략',

# 불용어 처리(stopwords-ko.txt 기반)


In [17]:
import os
current_directory = os.getcwd()
print("현재 디렉토리:", current_directory)

현재 디렉토리: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/notebooks/01_preprocess


In [18]:
# 불용어 사전 기반 불용어 리스트 정리
f = open(STOPWORDS_DIR / 'stopwords-ko.txt', 'r', encoding="UTF-8")
st = f.readlines()
f.close()

stw = []
for i in range(0, len(st)):
    stw.append(st[i].rstrip('\n'))

stw

['!',
 '"',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '...',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ';',
 '<',
 '=',
 '>',
 '?',
 '@',
 '\\',
 '^',
 '_',
 '`',
 '|',
 '~',
 '·',
 '—',
 '——',
 '‘',
 '’',
 '“',
 '”',
 '…',
 '、',
 '。',
 '〈',
 '〉',
 '《',
 '》',
 '가',
 '가까스로',
 '가령',
 '각',
 '각각',
 '각자',
 '각종',
 '갖고말하자면',
 '같다',
 '같이',
 '개의치않고',
 '거니와',
 '거바',
 '거의',
 '것',
 '것과 같이',
 '것들',
 '게다가',
 '게우다',
 '겨우',
 '견지에서',
 '결과에 이르다',
 '결국',
 '결론을 낼 수 있다',
 '겸사겸사',
 '고려하면',
 '고로',
 '곧',
 '공동으로',
 '과',
 '과연',
 '관계가 있다',
 '관계없이',
 '관련이 있다',
 '관하여',
 '관한',
 '관해서는',
 '구',
 '구체적으로',
 '구토하다',
 '그',
 '그들',
 '그때',
 '그래',
 '그래도',
 '그래서',
 '그러나',
 '그러니',
 '그러니까',
 '그러면',
 '그러므로',
 '그러한즉',
 '그런 까닭에',
 '그런데',
 '그런즉',
 '그럼',
 '그럼에도 불구하고',
 '그렇게 함으로써',
 '그렇지',
 '그렇지 않다면',
 '그렇지 않으면',
 '그렇지만',
 '그렇지않으면',
 '그리고',
 '그리하여',
 '그만이다',
 '그에 따르는',
 '그위에',
 '그저',
 '그중에서',
 '그치지 않다',
 '근거로',
 '근거하여',
 '기대여',
 '기점으로',
 '기준으로',
 '기타',
 '까닭으로',
 '까악',
 '까지',
 '까지 미치다',
 '까지도

In [19]:
# 각 문서의 제목과 본문 또는 댓글등에서 불용어 제거

for word in stw:
    for i in range(0, len(title_token_noun)) :
        # 리스트에 불용어가 있을 경우 제거
        while word in title_token_noun[i] :
            title_token_noun[i].remove(word)
    for j in range(0, len(document_token_noun)) : 
        while word in document_token_noun[j] :
            document_token_noun[j].remove(word)
    for l in range(0, len(comment_token_noun)) : 
        while word in comment_token_noun[l] :
            comment_token_noun[l].remove(word)

In [ ]:
# 문서파일 docs로 적용하여 각각의 불용어 제거
df['title_token_list_pos'] = title_token_list # 형태소와 품사 리스트
df['title_token_noun'] = title_token_noun # 명사 리스트

df['document_token_list_pos'] = document_token_list
df['document_token_noun'] = document_token_noun

df['comment_token_list_pos'] = comment_token_list
df['comment_token_noun'] = comment_token_noun

In [21]:
# 불용어 처리 후 최종 파일 저장 및 불러오기
import pickle

f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press.pkl", 'wb')
pickle.dump(df, f)
f.close()

In [23]:
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press.pkl", 'rb')
data = pickle.load(f)
f.close()
data

,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_list_pos,title_token_noun,document_token_list_pos,document_token_noun,comment_token_list_pos,comment_token_noun
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4,"[(2026, SN), (학년도, NNG), (의과, NNG), (대학, NNG),...","[학년도, 의과, 대학, 수시, 모집, 교과, 일반, 전형]","[(인제대, NNP), (는, JX), (작년, NNG), (내신, NNG), (성...","[인제대, 작년, 내신, 성적, 반영, 국영수, 과학, 상위, 과목, 반영, 학교,...","[(잘, MAG), (읽, VV), (어, EC), (보, VX), (었, EP),...","[한눈, 전체, 내용, 부연, 설명, 감사, 객관, 인칭, 시점, 보믄, 파라미터,..."
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4,"[(추가, NNG), (의대, NNG), (증원, NNG), (이야기, NNG), ...","[추가, 의대, 증원, 이야기]","[(최근, NNG), (에, JKB), (뉴스, NNG), (보, VV), (니까,...","[최근, 뉴스, 의대, 추가, 증원, 사실, 검색, 추천, 생물, 노용관, 이론, ...","[(올해, NNG), (는, JX), (일단, MAG), (유예, NNG), (되,...","[올해, 유예, 취소, 유예, 회의, 내년, 증원, 결정]"
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4,"[(sky, SL), (질문, NNG)]",[질문],"[(언미생사, NNP), (all, SL), (백분위, NNG), (96, SN),...","[언미생사, 백분위, 영어, 고정, 연고, 유리, 올해, 재수, 의대, 증원, 폐지...","[(뭐, NP), (가, JKS), (기준, NNG), (이, VCP), (ᆫ지, ...","[기준, 수능, 생각, 스카이, 유리, 생각, 이유, 성적, 유리, 수미, 잡이, ..."
3,수시 6장 머리터지기 시작했어요,글쓰기 전 선검색 바랍니다 미취학글 과외구인글 아이디 추천글 댓글 달린 글 삭제시 ...,1,11,[{'comment_content': '전체내신나오고 아이의 생기부가 어느정도 파악...,cafe,2025-06-30,4,"[(수시, NNG), (6, SN), (장, NNG), (머리, NNG), (터지,...","[수시, 머리, 시작]","[(글, NNG), (쓰, VV), (기, ETN), (전, NNG), (선, NN...","[검색, 취학, 과외, 구인, 아이디, 추천, 댓글, 삭제, 회원, 강등, 최하, ...","[(전체, NNG), (내신, NNG), (나오, VV), (고, EC), (아이,...","[전체, 내신, 생기, 정도, 파악, 객관, 현재, 위치, 성향, 종합, 교과, 논..."
4,고3 수시 대학 라인 문제,고1 고3 필수 학종 합격 가이드 1 학생 학부모 여부 본인및 자녀의 학년 예 고3...,1,2,"[{'comment_content': '같은 고민 입니다 어렵네요', 'commen...",cafe,2025-06-30,4,"[(고, NNG), (3, SN), (수시, NNG), (대학, NNG), (라인,...","[수시, 대학, 라인, 문제]","[(고, NNG), (1, SN), (고, NNG), (3, SN), (필수, NN...","[필수, 학종, 합격, 가이드, 학생, 학부모, 본인, 자녀, 학년, 재학, 카페,...","[(같, VA), (은, ETM), (고민, NNG), (이, VCP), (ᆸ니다,...","[고민, 안녕, 회원님유니브클래스, 카페, 회원, 축하, 처음, 유니브클래스, 카페..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3721,서울대 수시 합격생 11 가 미등록,1 정치 및 종교 저작권 위반 타학원 및 관계자에 대한 지나친 표현의 글은 작성금지...,0,12,"[{'comment_content': '다들 의대가려고 그런가바요', 'commen...",cafe,2024-01-02,1,"[(서울대, NNP), (수시, NNG), (합격, NNG), (생, XSN), (...","[서울대, 수시, 합격, 등록]","[(1, SN), (정치, NNG), (및, MAG), (종교, NNG), (저작,...","[정치, 종교, 저작, 위반, 학원, 관계자, 표현, 작성, 금지, 사이트, 블로그...","[(다, NNG), (들, XSN), (의대, NNG), (가, VV), (려고, ...","[의대, 의대, 영향, 의대, 생각, 자연, 학생, 의대, 중복, 지원, 경우, 생..."
3722,의대 증원 관련 기사입니다,단독 이과 수험생 62 교차지원 고려 의대증원 노린 반수 늘 듯 동아일보 단독 이과...,0,4,"[{'comment_content': '점점 의대중심의 세상이 되어가네요', 'co...",cafe,2024-01-01,1,"[(의대, NNG), (증원, NNG), (관련, NNG), (기사, NNG), (...","[의대, 증원, 관련, 기사]","[(단독, NNG), (이과, NNG), (수험, NNG), (생, XSN), (6...","[단독, 이과, 수험, 교차, 지원, 고려, 의대, 증원, 반수, 동아일보, 단독,...","[(점점, MAG), (의대, NNG), (중심, NNG), (의, JKG), (세...","[의대, 중심, 세상, 나중, 의대, 취급, 다양, 직업, 존재, 의대, 의대, 열..."
3723,부끄러운 얘기지만 치과는 여러 곳 가보세요,부끄러운 얘기지만 치과는 여러 곳 가보세요 치과계 폭리 다룬 책 펴낸 치과의사 김광...,6,5,[{'comment_content': '20 30년 정도 오래된 치과에 가는 것을 ...,cafe,2024-01-01,1,"[(부끄럽, VA-I), (은, ETM), (얘기, NNG), (이, VCP), (...","[얘기, 치과]","[(부끄럽, VA-I), (은, ETM), (얘기, NNG), (이, VCP), (...","[얘기, 치과, 치과, 폭리, 치과, 의사, 김광수, 치과, 폭리, 지적, 저자, ...","[(20, SN), (30, SN), (년, NNB), (정도, NNG), (오래,...","[정도, 치과, 추천, 조언, 대형, 치과, 상업, 자유, 갑진, 새해, 시작, 새..."
3724,2024년신년사 서울아산병원장 청라 개원 관련 여러번 언급,뉴스 행사2024년 신년사 서울아산병원장 박승일2024 01 01 2024년 갑진년...,54,18,[{'comment_content': '중입자치료기도입은 서울에 한다는걸까요 청라에...,cafe,2024-01-01,1,"[(2024, SN), (년, NNB), (신년사, NNG), (서울아산병원, NN...","[신년사, 서울아산병원, 청라, 개원, 관련, 언급]","[(뉴스, NNG), (행사, NNG), (2024, SN), (년, NNB), (...","[뉴스, 행사, 신년사, 서울아산병원, 박승일, 갑진, 새해, 서울아산병원, 가족,...","[(중입자, NNP), (치료기, NNG), (도입, NNG), (은, JX), (...","[중입자, 치료기, 도입, 서울, 청라

In [24]:
data_drop_list = data.drop(columns=['title_token_list_pos', 'document_token_list_pos', 'comment_token_list_pos'])
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl", 'wb')
pickle.dump(data_drop_list, f)
f.close()

In [25]:
f = open(DATA_CAFE_ONLY / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl", 'rb')
d = pickle.load(f)
f.close()
d

,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_noun,document_token_noun,comment_token_noun
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4,"[학년도, 의과, 대학, 수시, 모집, 교과, 일반, 전형]","[인제대, 작년, 내신, 성적, 반영, 국영수, 과학, 상위, 과목, 반영, 학교,...","[한눈, 전체, 내용, 부연, 설명, 감사, 객관, 인칭, 시점, 보믄, 파라미터,..."
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4,"[추가, 의대, 증원, 이야기]","[최근, 뉴스, 의대, 추가, 증원, 사실, 검색, 추천, 생물, 노용관, 이론, ...","[올해, 유예, 취소, 유예, 회의, 내년, 증원, 결정]"
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4,[질문],"[언미생사, 백분위, 영어, 고정, 연고, 유리, 올해, 재수, 의대, 증원, 폐지...","[기준, 수능, 생각, 스카이, 유리, 생각, 이유, 성적, 유리, 수미, 잡이, ..."
3,수시 6장 머리터지기 시작했어요,글쓰기 전 선검색 바랍니다 미취학글 과외구인글 아이디 추천글 댓글 달린 글 삭제시 ...,1,11,[{'comment_content': '전체내신나오고 아이의 생기부가 어느정도 파악...,cafe,2025-06-30,4,"[수시, 머리, 시작]","[검색, 취학, 과외, 구인, 아이디, 추천, 댓글, 삭제, 회원, 강등, 최하, ...","[전체, 내신, 생기, 정도, 파악, 객관, 현재, 위치, 성향, 종합, 교과, 논..."
4,고3 수시 대학 라인 문제,고1 고3 필수 학종 합격 가이드 1 학생 학부모 여부 본인및 자녀의 학년 예 고3...,1,2,"[{'comment_content': '같은 고민 입니다 어렵네요', 'commen...",cafe,2025-06-30,4,"[수시, 대학, 라인, 문제]","[필수, 학종, 합격, 가이드, 학생, 학부모, 본인, 자녀, 학년, 재학, 카페,...","[고민, 안녕, 회원님유니브클래스, 카페, 회원, 축하, 처음, 유니브클래스, 카페..."
...,...,...,...,...,...,...,...,...,...,...,...
3721,서울대 수시 합격생 11 가 미등록,1 정치 및 종교 저작권 위반 타학원 및 관계자에 대한 지나친 표현의 글은 작성금지...,0,12,"[{'comment_content': '다들 의대가려고 그런가바요', 'commen...",cafe,2024-01-02,1,"[서울대, 수시, 합격, 등록]","[정치, 종교, 저작, 위반, 학원, 관계자, 표현, 작성, 금지, 사이트, 블로그...","[의대, 의대, 영향, 의대, 생각, 자연, 학생, 의대, 중복, 지원, 경우, 생..."
3722,의대 증원 관련 기사입니다,단독 이과 수험생 62 교차지원 고려 의대증원 노린 반수 늘 듯 동아일보 단독 이과...,0,4,"[{'comment_content': '점점 의대중심의 세상이 되어가네요', 'co...",cafe,2024-01-01,1,"[의대, 증원, 관련, 기사]","[단독, 이과, 수험, 교차, 지원, 고려, 의대, 증원, 반수, 동아일보, 단독,...","[의대, 중심, 세상, 나중, 의대, 취급, 다양, 직업, 존재, 의대, 의대, 열..."
3723,부끄러운 얘기지만 치과는 여러 곳 가보세요,부끄러운 얘기지만 치과는 여러 곳 가보세요 치과계 폭리 다룬 책 펴낸 치과의사 김광...,6,5,[{'comment_content': '20 30년 정도 오래된 치과에 가는 것을 ...,cafe,2024-01-01,1,"[얘기, 치과]","[얘기, 치과, 치과, 폭리, 치과, 의사, 김광수, 치과, 폭리, 지적, 저자, ...","[정도, 치과, 추천, 조언, 대형, 치과, 상업, 자유, 갑진, 새해, 시작, 새..."
3724,2024년신년사 서울아산병원장 청라 개원 관련 여러번 언급,뉴스 행사2024년 신년사 서울아산병원장 박승일2024 01 01 2024년 갑진년...,54,18,[{'comment_content': '중입자치료기도입은 서울에 한다는걸까요 청라에...,cafe,2024-01-01,1,"[신년사, 서울아산병원, 청라, 개원, 관련, 언급]","[뉴스, 행사, 신년사, 서울아산병원, 박승일, 갑진, 새해, 서울아산병원, 가족,...","[중입자, 치료기, 도입, 서울, 청라, 도입, 변화, 혁신, 미래, 청라, 도입,..."


## 저장물 검증

카페 전처리 결과를 다시 읽어 02 통합 단계에서 필요한 명사 컬럼과 행 수를 확인합니다. 특히 통합 입력으로 사용하는 `_drop_list_pos.pkl` 파일이 존재하고, 제목·본문·댓글 명사 컬럼이 모두 리스트 형태인지 점검합니다.

In [26]:
# 02 통합 단계에서 사용하는 카페 PKL 저장물을 검증합니다.
cafe_validation_path = DATA_CAFE_ONLY / '의대증원_cafedata_total_estate_press_drop_list_pos.pkl'
validated_cafe = pd.read_pickle(cafe_validation_path)
required_cols = ["title", "doc", "comment_list", "section", "title_token_noun", "document_token_noun", "comment_token_noun"]
missing_cols = [col for col in required_cols if col not in validated_cafe.columns]
if missing_cols:
    raise KeyError(f"카페 전처리 저장물 누락 컬럼: {missing_cols}")

token_lengths = validated_cafe[["title_token_noun", "document_token_noun", "comment_token_noun"]].applymap(
    lambda value: len(value) if isinstance(value, list) else 0
)
print("검증 파일:", cafe_validation_path)
print("저장물 행 수:", len(validated_cafe))
print("section 분포:")
print(validated_cafe["section"].value_counts().sort_index())
print("빈 토큰 문서 수:", int((token_lengths.sum(axis=1) == 0).sum()))
validated_cafe.head(3)

검증 파일: /Users/hyerimchoi/DJD/datacrolling/20241983최혜림_20241285이서연_드론영상비정형데이터AI분석_중간과제/20241983최혜림_20241285이서연_드론비정형데이터AI분석/data/cafe_only/의대증원_cafedata_total_estate_press_drop_list_pos.pkl
저장물 행 수: 3726
section 분포:
section
1     671
2     650
3    1376
4    1029
Name: count, dtype: int64
빈 토큰 문서 수: 0


/var/folders/wg/2y3_zgls3jxc_lntn4_q23mm0000gn/T/ipykernel_65292/3024802642.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  token_lengths = validated_cafe[["title_token_noun", "document_token_noun", "comment_token_noun"]].applymap(


,title,doc,like,comment_cnt,comment_list,ch,date,section,title_token_noun,document_token_noun,comment_token_noun
0,2026학년도 의과대학 수시모집 교과 일반전형 3,인제대는 작년 내신성적을 반영할 때 국영수 과학 상위 2과목만 반영하는 산식으로 다...,16,6,[{'comment_content': '잘 읽어보았습니다 한눈에 전체 내용이 잘 들...,cafe,2025-06-29,4,"[학년도, 의과, 대학, 수시, 모집, 교과, 일반, 전형]","[인제대, 작년, 내신, 성적, 반영, 국영수, 과학, 상위, 과목, 반영, 학교,...","[한눈, 전체, 내용, 부연, 설명, 감사, 객관, 인칭, 시점, 보믄, 파라미터,..."
1,추가 의대증원 이야기 나오던데,최근에 뉴스 보니까 의대 또 추가증원한다는것 같던데 이거 사실인가요 어디서 본거 같...,0,1,[{'comment_content': '올해는 일단 유예됐는데 취소가 아니라 유예된...,cafe,2025-06-30,4,"[추가, 의대, 증원, 이야기]","[최근, 뉴스, 의대, 추가, 증원, 사실, 검색, 추천, 생물, 노용관, 이론, ...","[올해, 유예, 취소, 유예, 회의, 내년, 증원, 결정]"
2,sky 질문,언미생사 all 백분위 96인 1이고 영어는 고정 1인데 서연고 중에 어디가 제일 ...,0,3,[{'comment_content': '뭐가 기준인지는 모르겠지만 수능은 생각보다 ...,cafe,2025-06-29,4,[질문],"[언미생사, 백분위, 영어, 고정, 연고, 유리, 올해, 재수, 의대, 증원, 폐지...","[기준, 수능, 생각, 스카이, 유리, 생각, 이유, 성적, 유리, 수미, 잡이, ..."


section 4의 데이터 개수가 1029개로 블로그에서 다른 section에 비해 데이터 수가 현저히 적었던 section 4의 데이터 수를 보완합니다.
